In [47]:
# -- read in libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

In [48]:
# -- read in data
matches = pd.read_csv('matches_updated.csv') # -- Needs to be filtered
display(matches); print()

ALL_WRESTLERS = pd.read_csv('ALL_WRESTLERS.xls')
display(ALL_WRESTLERS)

teams = pd.read_csv('teams.xls')
display(teams); print()

duals = pd.read_csv('dual_meets.xls')
display(duals)

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name
0,280,141,2025-11-01,267.0,Raymond Adams,160.0,825.0,John Hildebrandt,98.0,False,LDEC,LDEC 3 - 2,JR,Duke,JR,Sacred Heart
1,283,197,2025-11-01,878.0,Kael Bennie,46.0,263.0,Angelo Posada,20.0,False,LDEC,LDEC 5 - 4,SO,Utah Valley,FR,Stanford
2,283,285,2025-11-01,879.0,Jack Forbes,62.0,429.0,Jackson Mankowski,182.0,True,WDEC,WDEC 10 - 3,SR,Utah Valley,SO,Stanford
3,284,125,2025-11-01,784.0,Koda Holeman,40.0,530.0,Jacob Macatangay,69.0,True,WMD,WMD 20 - 8,JR,Cal Poly,JR,Purdue
4,284,133,2025-11-01,785.0,Anthony Lucio,245.0,226.0,Blake Boarman,57.0,False,LDEC,LDEC 8 - 2,RSFR,Cal Poly,SR,Purdue
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5835,547,174,2026-02-23,399.0,Joshua Roe,237.0,814.0,Beau Lewis,158.0,False,LDEC,LDEC 5 - 3,SO,Presbyterian,FR,VMI
5836,547,184,2026-02-23,400.0,Reed Douglass,72.0,815.0,Andrew Barford,162.0,True,WTF,WTF5 22 - 5 6:45,JR,Presbyterian,FR,VMI
5837,547,197,2026-02-23,911.0,Toler Hornick,207.0,816.0,Toby Schoffstall,74.0,False,LTF,LTF5 15 - 0 1:05,SO,Presbyterian,JR,VMI
5838,547,149,2026-02-23,398.0,Rey Ortiz,245.0,811.0,Patrick Jordon,77.0,False,LFALL,LFALL 4:22,SO,Presbyterian,JR,VMI


,Team,Weight,Name,Class,Record
0,Air Force,125,"#28 Owens, TuckerST",SR,19 - 8
1,Air Force,125,"#157 Gonzalez, Nick",SR,0 - 1
2,Air Force,125,"#181 Tocci, Nico",JR,0 - 2
3,Air Force,125,"#212 Bohnsack, Brayden",FR,0 - 0
4,Air Force,133,"#121 Caprella, GavinST",JR,9 - 12
...,...,...,...,...,...
2640,Wyoming,197,"#3 Novak, JoeyST",JR,14 - 2
2641,Wyoming,197,"#88 Henry, GunnerRS",FR,6 - 3
2642,Wyoming,285,"#11 Carroll, ChristianST",SO,16 - 4
2643,Wyoming,285,"#182 McBride, Winston",SO,1 - 0


,team_id,name
0,1,Northern Iowa
1,2,South Dakota State
2,3,Drexel
3,4,Northern Illinois
4,5,Central Michigan
...,...,...
73,74,Indiana
74,75,Army West Point
75,76,Wyoming
76,77,Bellarmine


,dual_id,team1_id,team1_rank,team1_score,team2_id,team2_rank,team2_score,event_date
0,1,1,14,20,2,20,14,01/10/26
1,2,1,14,22,3,42,16,01/10/26
2,3,3,42,21,4,45,12,01/10/26
3,4,2,20,28,5,50,10,01/10/26
4,5,5,50,23,6,55,15,01/10/26
...,...,...,...,...,...,...,...,...
579,580,50,26,22,33,35,18,02/19/26
580,581,63,47,28,8,75,10,02/19/26
581,582,44,39,37,34,57,3,02/19/26
582,583,54,42,32,60,65,14,02/19/26


In [39]:
# -- Clean ALL_WRESTLERS(Contains wrestler class)
STATUS_CODES = ["ST", "RS", "RSFR", "RSSO", "RSJR"]

def clean_name(name_str):
    # Extract rank
    rank_match = re.search(r"#(\d+)", name_str)
    rank = int(rank_match.group(1)) if rank_match else None

    # Remove rank
    name_only = re.sub(r"#\d+\s*", "", name_str)

    # Remove trailing known status ONLY if attached
    for status in STATUS_CODES:
        if name_only.endswith(status):
            name_only = name_only[:-len(status)]
            break

    name_only = name_only.strip()

    # Split last and first
    if "," in name_only:
        last, first = name_only.split(",", 1)
        return rank, last.strip(), first.strip()

    return rank, None, None


#🚀 Apply It To Your DataFrame
all_wrestler_info = ALL_WRESTLERS.copy()
all_wrestler_info[["Rank", "Last", "First"]] = all_wrestler_info["Name"].apply(
    lambda x: pd.Series(clean_name(x))
)

all_wrestler_info['name'] = all_wrestler_info['First'] + " " + all_wrestler_info['Last']
all_wrestler_info.loc[all_wrestler_info['name'] == "JD Perez", 'name'] = "Jesse Perez" # edge case

all_wrestler_info.query('name == "Gabe Arnold"')

,Team,Weight,Name,Class,Record,Rank,Last,First,name
1056,Iowa,197,"#14 Arnold, GabeST",SO,13 - 5,14,Arnold,Gabe,Gabe Arnold


In [37]:
teams.query('name == "Ohio State"')

,team_id,name
40,41,Ohio State


In [38]:
duals.query('team1_id == 41 or team2_id == 41')

,dual_id,team1_id,team1_rank,team1_score,team2_id,team2_rank,team2_score,event_date
41,42,41,4,41,40,24,3,01/04/26
60,61,41,4,21,39,6,13,12/21/25
61,62,41,4,34,53,37,9,12/21/25
128,129,41,4,26,21,7,10,12/12/25
159,160,41,4,41,77,54,3,12/03/25
183,184,41,4,44,34,62,0,11/20/25
187,188,41,4,27,17,2,12,11/16/25
188,189,41,4,33,19,3,3,11/16/25
207,208,41,4,29,14,8,6,11/15/25
220,221,41,4,33,76,26,6,11/15/25


In [40]:
matches.query('home_name == "Gabe Arnold" or away_name == "Gabe Arnold"')

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name
1029,178,184,2025-11-21,992.0,Gabe Arnold,15.0,301.0,Chase Kranitz,38.0,True,WMD,WMD 15 - 5,SO,Iowa,JR,Pittsburgh
1532,128,197,2025-12-12,992.0,Gabe Arnold,15.0,997.0,Kade Rule,48.0,True,WMD,WMD 17 - 4,SO,Iowa,FR,Chattanooga
1542,127,197,2025-12-12,992.0,Gabe Arnold,15.0,878.0,Kael Bennie,46.0,True,WDEC,WDEC 4 - 2,SO,Iowa,SO,Utah Valley
3268,324,174,2026-01-16,992.0,Gabe Arnold,17.0,125.0,Levi Haines,1.0,False,LDEC,LDEC 4 - 2,SO,Iowa,SR,Penn State
3578,373,184,2026-01-23,222.0,Silas Allred,7.0,992.0,Gabe Arnold,13.0,False,LDEC,LDEC 4 - 1,SR,Nebraska,SO,Iowa
4055,413,184,2026-01-30,992.0,Gabe Arnold,13.0,156.0,Max McEnelly,3.0,False,LDEC,LDEC 4 - 1,SO,Iowa,SO,Minnesota
4523,511,184,2026-02-06,449.0,Dylan Fishback,8.0,992.0,Gabe Arnold,10.0,True,WSV,WSV-1 4 - 1,JR,Ohio State,SO,Iowa
4900,500,184,2026-02-08,291.0,Ryan Boucher,128.0,992.0,Gabe Arnold,10.0,False,LTF,LTF5 20 - 4 7:00,SR,Michigan State,SO,Iowa
4969,470,184,2026-02-13,992.0,Gabe Arnold,10.0,281.0,Brock Mantanona,11.0,True,WTB,WTB-1 3 - 2,SO,Iowa,RSFR,Michigan
5251,444,184,2026-02-15,232.0,James Rowley,32.0,992.0,Gabe Arnold,10.0,False,LDEC,LDEC 4 - 1,JR,Purdue,SO,Iowa


In [53]:
def compute_season_statistics(matches_path='matches_updated.csv', 
                             all_wrestlers_path='ALL_WRESTLERS.xls',
                             teams_path='teams.xls',
                             duals_path='dual_meets.xls'):
    """
    Compute season-long statistics for every wrestler.
    Filters out DQ/INJ/FORFEIT matches.
    Returns a DataFrame with one row per wrestler and key stats.
    """
    
    print("="*80)
    print("COMPUTING SEASON STATISTICS FOR ALL WRESTLERS")
    print("="*80)
    
    # ============================================
    # LOAD DATA
    # ============================================
    print("\n📂 Loading data...")
    
    matches = pd.read_csv(matches_path, parse_dates=['event_date'])
    all_wrestlers = pd.read_csv(all_wrestlers_path)
    teams = pd.read_csv(teams_path)
    duals = pd.read_csv(duals_path, parse_dates=['event_date'])
    
    print(f"   ✓ Matches: {len(matches)}")
    print(f"   ✓ All Wrestlers: {len(all_wrestlers)}")
    
    # ============================================
    # CLEAN ALL_WRESTLERS
    # ============================================
    print("\n🧹 Cleaning ALL_WRESTLERS data...")
    
    STATUS_CODES = ["ST", "RS", "RSFR", "RSSO", "RSJR", "RSSR"]
    
    def clean_name(name_str):
        if pd.isna(name_str):
            return None, None, None, None
        
        # Extract rank
        rank_match = re.search(r"#(\d+)", name_str)
        rank = int(rank_match.group(1)) if rank_match else None
        
        # Remove rank
        name_only = re.sub(r"#\d+\s*", "", name_str)
        
        # Remove trailing known status
        for status in STATUS_CODES:
            if name_only.endswith(status):
                name_only = name_only[:-len(status)]
                break
        
        name_only = name_only.strip()
        
        # Split last and first
        if "," in name_only:
            last, first = name_only.split(",", 1)
            return rank, last.strip(), first.strip(), name_only
        
        return rank, None, None, name_only
    
    # Apply cleaning
    all_wrestlers[["Rank", "Last", "First", "CleanedName"]] = all_wrestlers["Name"].apply(
        lambda x: pd.Series(clean_name(x))
    )
    
    all_wrestlers['name'] = all_wrestlers['First'] + " " + all_wrestlers['Last']
    all_wrestlers['name'] = all_wrestlers['name'].str.strip()
    
    # Fix edge cases
    all_wrestlers.loc[all_wrestlers['name'] == "JD Perez", 'name'] = "Jesse Perez"
    
    # Create wrestler info mapping
    wrestler_info = all_wrestlers[['name', 'Team', 'Weight', 'Class', 'Rank']].drop_duplicates(subset=['name'])
    print(f"   ✓ Cleaned {len(wrestler_info)} unique wrestlers")
    
    # ============================================
    # FILTER OUT DQ/INJ/FORFEIT MATCHES
    # ============================================
    print("\n🔍 Filtering matches...")
    
    exclude_types = ['DQ', 'INJ', 'FOR', 'DFF']
    matches_filtered = matches[~matches['win_type'].str.contains('|'.join(exclude_types), na=False)].copy()
    
    print(f"   ✓ Before filtering: {len(matches)} matches")
    print(f"   ✓ After filtering: {len(matches_filtered)} matches")
    print(f"   ✓ Removed {len(matches) - len(matches_filtered)} matches with DQ/INJ/FORFEIT")
    
    # ============================================
    # GET TEAM RANKS
    # ============================================
    print("\n🏆 Getting team ranks...")
    
    team_name_to_id = dict(zip(teams['name'], teams['team_id']))
    
    def get_latest_team_rank(team_name):
        if pd.isna(team_name):
            return None
        
        team_id = team_name_to_id.get(team_name)
        if team_id is None:
            return None
        
        # Find most recent dual where team appears
        team_duals = duals[(duals['team1_id'] == team_id) | (duals['team2_id'] == team_id)]
        
        if len(team_duals) == 0:
            return None
        
        latest_dual = team_duals.sort_values('event_date', ascending=False).iloc[0]
        
        if latest_dual['team1_id'] == team_id:
            return latest_dual['team1_rank']
        else:
            return latest_dual['team2_rank']
    
    # ============================================
    # HELPER FUNCTIONS FOR STATS
    # ============================================
    
    def get_base_win_type(win_type):
        if pd.isna(win_type):
            return None
        wt = str(win_type).upper().strip()
        if "FALL" in wt:
            return "FALL"
        elif "TF" in wt:
            return "TF"
        elif "MD" in wt:
            return "MD"
        elif "DEC" in wt or "SV" in wt or "TB" in wt:
            return "DEC"
        return "OTHER"
    
    def extract_points(result, win_type):
        if pd.isna(win_type) or pd.isna(result):
            return None, None, False
        
        wt = str(win_type).upper().strip()
        
        if any(x in wt for x in ["DEC", "MD", "TF", "SV", "TB"]):
            try:
                parts = str(result).split()
                scores = [int(x) for x in parts if x.isdigit()]
                if len(scores) >= 2:
                    return scores[0], scores[1], "TF" in wt
            except:
                pass
        
        return None, None, False
    
    # ============================================
    # BUILD WRESTLER MATCH RECORDS
    # ============================================
    print("\n🔄 Building wrestler match records...")
    
    records = []
    
    for _, row in matches_filtered.iterrows():
        # Create unique wrestler key (using name + team to handle duplicates)
        home_key = f"{row['home_name']}_{row['home_team_name']}"
        away_key = f"{row['away_name']}_{row['away_team_name']}"
        
        base_type = get_base_win_type(row['win_type'])
        score1, score2, is_tf = extract_points(row['Result'], row['win_type'])
        
        # Home wrestler
        home_won = row['home_win']
        records.append({
            'wrestler_key': home_key,
            'wrestler_name': row['home_name'],
            'team_name': row['home_team_name'],
            'weight_class': row['weight_class'],
            'win': home_won,
            'win_type': base_type,
            'points_scored': score1 if home_won and score1 is not None else (score2 if not home_won and score2 is not None else None),
            'point_diff': (15 if is_tf and home_won else -15 if is_tf else 
                          (score1 - score2 if home_won and score1 is not None else 
                           (score2 - score1 if not home_won and score2 is not None else None))),
            'opponent_rank': row['away_rank']
        })
        
        # Away wrestler
        away_won = not home_won
        records.append({
            'wrestler_key': away_key,
            'wrestler_name': row['away_name'],
            'team_name': row['away_team_name'],
            'weight_class': row['weight_class'],
            'win': away_won,
            'win_type': base_type,
            'points_scored': score1 if away_won and score1 is not None else (score2 if not away_won and score2 is not None else None),
            'point_diff': (15 if is_tf and away_won else -15 if is_tf else 
                          (score1 - score2 if away_won and score1 is not None else 
                           (score2 - score1 if not away_won and score2 is not None else None))),
            'opponent_rank': row['home_rank']
        })
    
    wrestler_matches = pd.DataFrame(records)
    print(f"   ✓ Created {len(wrestler_matches)} wrestler-match records")
    
    # ============================================
    # AGGREGATE SEASON STATISTICS
    # ============================================
    print("\n📊 Aggregating season statistics...")
    
    # Split into groups for processing
    season_stats_list = []
    
    for wrestler_key in wrestler_matches['wrestler_key'].unique():
        wrestler_data = wrestler_matches[wrestler_matches['wrestler_key'] == wrestler_key]
        
        # Get basic info
        wrestler_name = wrestler_data.iloc[0]['wrestler_name']
        team_name = wrestler_data.iloc[0]['team_name']
        
        # Match counts
        total_matches = len(wrestler_data)
        total_wins = wrestler_data['win'].sum()
        total_losses = total_matches - total_wins
        
        # Win types
        wins_data = wrestler_data[wrestler_data['win']]
        tf_wins = len(wins_data[wins_data['win_type'] == 'TF'])
        md_wins = len(wins_data[wins_data['win_type'] == 'MD'])
        fall_wins = len(wins_data[wins_data['win_type'] == 'FALL'])
        dec_wins = len(wins_data[wins_data['win_type'] == 'DEC'])
        bonus_wins = tf_wins + md_wins + fall_wins
        
        # Points (excluding matches with no scores)
        scored_data = wrestler_data[wrestler_data['points_scored'].notna()]
        total_points_scored = scored_data['points_scored'].sum()
        avg_points_scored = total_points_scored / len(scored_data) if len(scored_data) > 0 else 0
        
        # Point differential
        diff_data = wrestler_data[wrestler_data['point_diff'].notna()]
        total_point_diff = diff_data['point_diff'].sum()
        avg_point_diff = total_point_diff / len(diff_data) if len(diff_data) > 0 else 0
        std_point_diff = diff_data['point_diff'].std() if len(diff_data) > 1 else 0
        
        # Opponent rank
        rank_data = wrestler_data[wrestler_data['opponent_rank'].notna()]
        avg_opponent_rank = rank_data['opponent_rank'].mean() if len(rank_data) > 0 else 0
        
        # Get wrestler rank from all_wrestlers
        wrestler_rank_info = wrestler_info[wrestler_info['name'] == wrestler_name]
        rank = wrestler_rank_info.iloc[0]['Rank'] if len(wrestler_rank_info) > 0 else None
        
        # Get team rank
        team_rank = get_latest_team_rank(team_name)
        
        # Get weight classes (could be multiple)
        weight_classes = sorted(wrestler_data['weight_class'].unique())
        
        season_stats_list.append({
            'wrestler_name': wrestler_name,
            'team_name': team_name,
            'rank': rank,
            'team_rank': team_rank,
            'weight_classes': str(weight_classes),
            'total_matches': total_matches,
            'total_wins': total_wins,
            'total_losses': total_losses,
            'win_rate': round(total_wins / total_matches * 100, 1) if total_matches > 0 else 0,
            'bonus_wins': bonus_wins,
            'bonus_rate': round(bonus_wins / total_matches * 100, 1) if total_matches > 0 else 0,
            'tf_wins': tf_wins,
            'md_wins': md_wins,
            'fall_wins': fall_wins,
            'dec_wins': dec_wins,
            'avg_opponent_rank': round(avg_opponent_rank, 2),
            'avg_points_scored': round(avg_points_scored, 2),
            'avg_point_diff': round(avg_point_diff, 2),
            'std_point_diff': round(std_point_diff, 2)
        })
    
    season_stats = pd.DataFrame(season_stats_list)
    season_stats = season_stats.sort_values('total_matches', ascending=False).reset_index(drop=True)
    
    print(f"   ✓ Aggregated stats for {len(season_stats)} wrestlers")
    
    # ============================================
    # TEST CASES VERIFICATION
    # ============================================
    print("\n" + "="*80)
    print("TEST CASES VERIFICATION")
    print("="*80)
    
    test_results = []
    
    # Drake Ayala
    drake = season_stats[season_stats['wrestler_name'] == 'Drake Ayala']
    if len(drake) > 0:
        drake = drake.iloc[0]
        print("\n🔍 DRAKE AYALA:")
        print(f"   Total matches: {drake['total_matches']} (expected 16)")
        print(f"   Rank: {drake['rank']} (expected 10)")
        print(f"   Team rank: {drake['team_rank']} (expected ~5)")
        print(f"   Win rate: {drake['win_rate']}%")
        print(f"   Bonus rate: {drake['bonus_rate']}%")
        
        test_results.append(['Drake Ayala', 'total_matches', drake['total_matches'], 16])
        test_results.append(['Drake Ayala', 'rank', drake['rank'], 10])
        test_results.append(['Drake Ayala', 'team_rank', drake['team_rank'], 5])
    
    # Nick Feldman
    nick = season_stats[season_stats['wrestler_name'] == 'Nick Feldman']
    if len(nick) > 0:
        nick = nick.iloc[0]
        print("\n🔍 NICK FELDMAN:")
        print(f"   Total matches: {nick['total_matches']} (expected 17)")
        print(f"   Rank: {nick['rank']} (expected 5)")
        print(f"   Team rank: {nick['team_rank']} (expected 2)")
        print(f"   Win rate: {nick['win_rate']}%")
        print(f"   Bonus rate: {nick['bonus_rate']}%")
        
        test_results.append(['Nick Feldman', 'total_matches', nick['total_matches'], 17])
        test_results.append(['Nick Feldman', 'rank', nick['rank'], 5])
        test_results.append(['Nick Feldman', 'team_rank', nick['team_rank'], 2])
    
    # Gabe Arnold
    gabe = season_stats[season_stats['wrestler_name'] == 'Gabe Arnold']
    if len(gabe) > 0:
        gabe = gabe.iloc[0]
        print("\n🔍 GABE ARNOLD:")
        print(f"   Total matches: {gabe['total_matches']} (expected 11)")
        print(f"   Weight classes: {gabe['weight_classes']} (expected [174, 184, 197])")
        print(f"   Rank: {gabe['rank']} (expected 14)")
        print(f"   Win rate: {gabe['win_rate']}%")
        print(f"   Bonus rate: {gabe['bonus_rate']}%")
        
        test_results.append(['Gabe Arnold', 'total_matches', gabe['total_matches'], 11])
        test_results.append(['Gabe Arnold', 'weight_classes', gabe['weight_classes'], '[174, 184, 197]'])
        test_results.append(['Gabe Arnold', 'rank', gabe['rank'], 14])
    
    # Test results DataFrame
    test_df = pd.DataFrame(test_results, columns=['wrestler', 'metric', 'actual', 'expected'])
    test_df['status'] = test_df.apply(
        lambda row: '✅ PASS' if row['actual'] == row['expected'] or 
                   (row['metric'] == 'team_rank' and abs(row['actual'] - row['expected']) <= 2)
                   else '❌ FAIL', 
        axis=1
    )
    
    print("\n" + "="*60)
    print("TEST RESULTS SUMMARY")
    print("="*60)
    print(test_df.to_string(index=False))
    
    # ============================================
    # SAVE FINAL DATAFRAME
    # ============================================
    print("\n" + "="*80)
    print("✅ SEASON STATISTICS COMPLETE")
    print("="*80)
    print(f"\nFinal DataFrame shape: {season_stats.shape}")
    print(f"Columns: {season_stats.columns.tolist()}")
    print(f"\nFirst 5 rows:")
    print(season_stats.head())
    
    return season_stats

# Run it
season_stats = compute_season_statistics()
print("\n✅ Done! season_stats DataFrame contains stats for all wrestlers")

COMPUTING SEASON STATISTICS FOR ALL WRESTLERS

📂 Loading data...
   ✓ Matches: 5840
   ✓ All Wrestlers: 2645

🧹 Cleaning ALL_WRESTLERS data...


/tmp/ipykernel_1068/389356671.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  duals = pd.read_csv(duals_path, parse_dates=['event_date'])


   ✓ Cleaned 2642 unique wrestlers

🔍 Filtering matches...
   ✓ Before filtering: 5840 matches
   ✓ After filtering: 5632 matches
   ✓ Removed 208 matches with DQ/INJ/FORFEIT

🏆 Getting team ranks...

🔄 Building wrestler match records...
   ✓ Created 11264 wrestler-match records

📊 Aggregating season statistics...
   ✓ Aggregated stats for 1506 wrestlers

TEST CASES VERIFICATION

🔍 DRAKE AYALA:
   Total matches: 16 (expected 16)
   Rank: 10 (expected 10)
   Team rank: 5 (expected ~5)
   Win rate: 56.2%
   Bonus rate: 43.8%

🔍 NICK FELDMAN:
   Total matches: 17 (expected 17)
   Rank: 5 (expected 5)
   Team rank: 2 (expected 2)
   Win rate: 76.5%
   Bonus rate: 41.2%

🔍 GABE ARNOLD:
   Total matches: 11 (expected 11)
   Weight classes: [174, 184, 197] (expected [174, 184, 197])
   Rank: 14 (expected 14)
   Win rate: 63.6%
   Bonus rate: 27.3%

TEST RESULTS SUMMARY
    wrestler         metric          actual        expected status
 Drake Ayala  total_matches              16              1

In [55]:
season_stats.query('wrestler_name == "James Conway"')

,wrestler_name,team_name,rank,team_rank,weight_classes,total_matches,total_wins,total_losses,win_rate,bonus_wins,bonus_rate,tf_wins,md_wins,fall_wins,dec_wins,avg_opponent_rank,avg_points_scored,avg_point_diff,std_point_diff
564,James Conway,Missouri,6,14,"[157, 165]",10,5,5,50.0,2,20.0,1,0,1,3,58.40,4.89,0.67,6.3
632,James Conway,Franklin & Marshall,6,67,[184],9,9,0,100.0,8,88.9,5,1,2,1,157.56,16.43,13.29,3.4


In [56]:
matches.query('home_wrestler == "James Conw"')

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name
0,280,141,2025-11-01,267.0,Raymond Adams,160.0,825.0,John Hildebrandt,98.0,False,LDEC,LDEC 3 - 2,JR,Duke,JR,Sacred Heart
1,283,197,2025-11-01,878.0,Kael Bennie,46.0,263.0,Angelo Posada,20.0,False,LDEC,LDEC 5 - 4,SO,Utah Valley,FR,Stanford
2,283,285,2025-11-01,879.0,Jack Forbes,62.0,429.0,Jackson Mankowski,182.0,True,WDEC,WDEC 10 - 3,SR,Utah Valley,SO,Stanford
3,284,125,2025-11-01,784.0,Koda Holeman,40.0,530.0,Jacob Macatangay,69.0,True,WMD,WMD 20 - 8,JR,Cal Poly,JR,Purdue
4,284,133,2025-11-01,785.0,Anthony Lucio,245.0,226.0,Blake Boarman,57.0,False,LDEC,LDEC 8 - 2,RSFR,Cal Poly,SR,Purdue
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5835,547,174,2026-02-23,399.0,Joshua Roe,237.0,814.0,Beau Lewis,158.0,False,LDEC,LDEC 5 - 3,SO,Presbyterian,FR,VMI
5836,547,184,2026-02-23,400.0,Reed Douglass,72.0,815.0,Andrew Barford,162.0,True,WTF,WTF5 22 - 5 6:45,JR,Presbyterian,FR,VMI
5837,547,197,2026-02-23,911.0,Toler Hornick,207.0,816.0,Toby Schoffstall,74.0,False,LTF,LTF5 15 - 0 1:05,SO,Presbyterian,JR,VMI
5838,547,149,2026-02-23,398.0,Rey Ortiz,245.0,811.0,Patrick Jordon,77.0,False,LFALL,LFALL 4:22,SO,Presbyterian,JR,VMI
